# Sampling Methods -- DS Interview Reference (R)

## Quick Index

| Goal | Method | Section |
| :--- | :--- | :--- |
| Split data into train/test without caret/tidymodels | Simple Random Sampling | SS1 |
| Preserve class proportions in a split | Stratified Sampling | SS2 |
| Sample efficiently from ordered/large data | Systematic Sampling | SS3 |
| Sample grouped data (sessions, users) | Cluster Sampling | SS4 |
| CI for any statistic without distributional assumptions | Bootstrap | SS5 |
| Detect influential observations / bias correction | Jackknife | SS6 |
| Handle class imbalance, weighted metrics | Weighted Sampling | SS7 |
| Stratified cross-validation without tidymodels | Stratified K-Fold | SS8 |
| A/B test significance and effect size | Two-Sample Bootstrap | SS9 |

---

## When to Reach for Each Method

| Interview signal | Method |
| :--- | :--- |
| "Split without a package", "reproducible split" | SS1 SRS |
| "Imbalanced classes", "preserve proportions" | SS2 Stratified split |
| "Large ordered dataset", "sample a log" | SS3 Systematic |
| "Sample by user/session", "grouped data" | SS4 Cluster |
| "CI without a stats package", "any statistic", "model uncertainty" | SS5 Bootstrap |
| "Which observations matter most?", "bias correction" | SS6 Jackknife |
| "Minority class", "weighted accuracy/metric" | SS7 Weighted |
| "Cross-validation without tidymodels", "k-fold" | SS8 Stratified k-fold |
| "A/B test", "compare two groups", "significance" | SS9 Two-Sample Bootstrap |

---

> **Interview note (R vs Python):** Base R already has `quantile()` for percentile
> bootstrap CIs (R's analog to `np.percentile`, no package needed). For t-critical
> values, base R's `qt()` gives exact values directly -- there is no need to
> pre-compute or approximate them the way the Python source does (`t(0.975, df=49)
> = qt(0.975, 49)`); this reference uses `qt()` directly throughout rather than
> hard-coded constants, since it costs nothing extra in R.

---
## SS1 -- Simple Random Sampling (SRS)

**Intuition:** Every unit has an equal probability of selection -- the "lottery"
model. The baseline all other methods compare against. In DS interviews this
appears most often disguised as: *"implement a train/test split without using a
modeling package."*

**Key properties:**
- Unbiased estimator of the population mean
- Variance decreases as 1/n
- `dplyr::slice_sample(prop = 1)` shuffles; slicing the result gives the split

**Confidence Interval:**

$$\bar{x} \pm z \cdot \frac{s}{\sqrt{n}} \quad (z = 1.96 \text{ for 95\%, valid for } n \geq 30)$$

---

> **Interview prompt:** *"Without using a modeling package, implement a function
> that splits a data frame into train and test sets given a test fraction. Ensure
> it is reproducible and verify there is no overlap between the two sets."*

In [ ]:
library(dplyr)

set.seed(42)

# ── Simulate dataset ──────────────────────────────────────────────────────────
N <- 10000
df_pop <- tibble::tibble(
  user_id = 1:N,
  revenue = round(rexp(N, rate = 1/50), 2)
)

# ── Core patterns ─────────────────────────────────────────────────────────────
n <- 200
sample_dplyr <- df_pop |> slice_sample(n = n)                       # dplyr -- no explicit replace= needed, default FALSE
sample_base  <- sample(df_pop$revenue, size = n, replace = FALSE)   # base R

# ── Interview task: train_test_split from scratch ─────────────────────────────
train_test_split <- function(df, test_frac = 0.2, seed = 42) {
  # Random train/test split without a modeling package.
  # Shuffles once then slices -- O(n), no index collisions.
  set.seed(seed)
  shuffled <- df |> slice_sample(prop = 1)
  n_test   <- as.integer(nrow(shuffled) * test_frac)
  test     <- shuffled |> slice(1:n_test)
  train    <- shuffled |> slice((n_test + 1):nrow(shuffled))
  list(train = train, test = test)
}

split <- train_test_split(df_pop, test_frac = 0.2, seed = 42)
train <- split$train; test <- split$test

# ── Verify: no overlap, correct sizes ────────────────────────────────────────
overlap <- intersect(train$user_id, test$user_id)
cat(sprintf("Train size : %s  (%.0f%%)\n", format(nrow(train), big.mark=","), nrow(train)/N*100))
cat(sprintf("Test size  : %s   (%.0f%%)\n", format(nrow(test), big.mark=","), nrow(test)/N*100))
cat(sprintf("Overlap    : %d rows  (must be 0)\n", length(overlap)))

# ── Confidence Interval (z-approximation, n >= 30) ───────────────────────────
x     <- train$revenue
x_bar <- mean(x)
se    <- sd(x) / sqrt(length(x))     # R's sd() is already n-1, no ddof= needed
z     <- 1.96
ci    <- c(x_bar - z * se, x_bar + z * se)

cat(sprintf("\nTrain mean  : %.2f\n", x_bar))
cat(sprintf("95%% CI      : (%.2f, %.2f)\n", ci[1], ci[2]))
cat(sprintf("True mean   : %.2f\n", mean(df_pop$revenue)))

**Common mistakes:**
- Forgetting `set.seed()` -- split is not reproducible. Unlike pandas' per-call `random_state=`, R's seed is **global state** set once with `set.seed()`; calling another random function afterward (even unrelated to your split) shifts the sequence, so set the seed immediately before the operation that needs it
- Relying on row position after filtering — tibbles don't carry a meaningful persistent index the way pandas `DataFrame`s do, which actually removes an entire category of pandas' `.iloc` vs `.loc` index-mismatch bugs; there's nothing equivalent to `reset_index(drop=True)` to remember
- Checking overlap by row content instead of ID -- two different rows can have identical values, same trap as pandas

---
## SS2 -- Stratified Sampling

**Intuition:** Divide the population into non-overlapping strata and sample
independently within each. Guarantees every subgroup is represented in
proportion to its size in the population. In DS interviews this is almost
always framed as a **class-imbalance problem** -- a minority class can
disappear entirely from a test set under plain SRS.

**Key properties:**
- Preserves class proportions in both train and test
- Variance is lower than SRS when strata have different means
- Disproportional allocation = deliberately oversample a minority class

**Confidence Interval:**

$$\bar{x}_{st} = \sum_h W_h \bar{x}_h, \quad SE = \sqrt{\sum_h W_h^2 \frac{s_h^2}{n_h}}$$

where $W_h = N_h / N$ is the stratum weight.

---

> **Interview prompt:** *"You have a dataset with 80/15/5 class imbalance.
> Implement a stratified train/test split without a modeling package that
> preserves those proportions. Verify the result."*

In [ ]:
library(dplyr)

set.seed(42)

# ── Simulate imbalanced classification dataset ────────────────────────────────
N      <- 5000
labels <- sample(c('class_A','class_B','class_C'), N, replace = TRUE, prob = c(0.80, 0.15, 0.05))
df <- tibble::tibble(feature = rnorm(N), label = labels)

cat("Population class distribution:\n")
print(round(prop.table(table(df$label)), 3))

# ── Interview task: stratified_split from scratch ─────────────────────────────
stratified_split <- function(df, target_col, test_frac = 0.2, seed = 42) {
  # Stratified train/test split preserving class proportions.
  # Samples test_frac from each stratum independently.
  set.seed(seed)
  train_parts <- list(); test_parts <- list()
  for (lbl in unique(df[[target_col]])) {
    grp      <- df |> filter(.data[[target_col]] == lbl)
    shuffled <- grp |> slice_sample(prop = 1)
    n_test   <- as.integer(nrow(shuffled) * test_frac)
    test_parts[[lbl]]  <- shuffled |> slice(1:n_test)
    train_parts[[lbl]] <- shuffled |> slice((n_test + 1):nrow(shuffled))
  }
  train <- bind_rows(train_parts) |> slice_sample(prop = 1)
  test  <- bind_rows(test_parts)  |> slice_sample(prop = 1)
  list(train = train, test = test)
}

split <- stratified_split(df, target_col = 'label', test_frac = 0.2, seed = 42)
train <- split$train; test <- split$test

# ── Verify proportions are preserved ──────────────────────────────────────────
comparison <- tibble::tibble(
  label      = sort(unique(df$label)),
  population = round(as.numeric(prop.table(table(df$label))[sort(unique(df$label))]), 3),
  train_p    = round(as.numeric(prop.table(table(train$label))[sort(unique(df$label))]), 3),
  test_p     = round(as.numeric(prop.table(table(test$label))[sort(unique(df$label))]), 3)
)
print(comparison)

# ── Disproportional allocation: oversample minority class ─────────────────────
target_n <- c(class_A = 200, class_B = 200, class_C = 200)   # equal allocation
parts <- list()
for (lbl in names(target_n)) {
  grp  <- df |> filter(label == lbl)
  n    <- target_n[[lbl]]
  repl <- nrow(grp) < n                                        # oversample if needed
  parts[[lbl]] <- grp |> slice_sample(n = n, replace = repl)
}
balanced <- bind_rows(parts)

cat("\nOversampled training set (equal allocation):\n")
print(table(balanced$label))

# ── Confidence Interval (weighted stratum formula) ────────────────────────────
N_total <- nrow(df)
grp_stats <- train |> group_by(label) |> summarise(
  n = n(), mean = mean(feature), var = var(feature), .groups = "drop"
)
N_h <- as.numeric(table(df$label)[grp_stats$label])
grp_stats <- grp_stats |> mutate(N_h = N_h, W_h = N_h / N_total)
strat_mean <- sum(grp_stats$W_h * grp_stats$mean)
strat_var  <- sum(grp_stats$W_h^2 * grp_stats$var / grp_stats$n)
ci <- c(strat_mean - 1.96*sqrt(strat_var), strat_mean + 1.96*sqrt(strat_var))
cat(sprintf("\nStratified mean : %.3f\n", strat_mean))
cat(sprintf("95%% CI          : (%.3f, %.3f)\n", ci[1], ci[2]))

**Stratified vs. SRS:**

| | Variance vs SRS | Requires | Best when |
| :--- | :--- | :--- | :--- |
| SRS | Baseline | Nothing | Balanced classes |
| Stratified | Lower | Known class labels | Imbalanced classes, small minorities |

**Common mistakes:**
- Looping over `unique(df$label)` and using `filter()` inside the loop is the clearest, most explicit approach in R — there is no `group_by() |> group_modify(~ slice_sample(...))` gotcha to worry about the way pandas' `groupby().apply()` silently drops the key in 3.x; group_modify() actually preserves grouping columns correctly, but the explicit loop above is shown for maximal clarity in an interview setting
- Forgetting to re-shuffle after `bind_rows()` -- train rows are blocked by class, which can cause issues with sequential models
- Oversampling test set -- only oversample the training set; keep test proportions natural

---
## SS3 -- Systematic Sampling

**Intuition:** Sort the population by a natural ordering, pick a random start
in [0, k), then select every k-th unit. Simpler than SRS for large ordered data
(one scan, no lookup table) and equally precise when the order is random.
The single failure mode: if the data has a periodic pattern with the same
period as k, the sample is catastrophically biased.

**Step size:** k = N / n (round to nearest integer). **Random start** in [0, k).

**Confidence Interval:** Approximate as SRS when autocorrelation is low.

$$\text{CI} \approx \bar{x} \pm 1.96 \cdot \frac{s}{\sqrt{n}}$$

---

> **Interview prompt:** *"You have a time-ordered log of 1 million server events.
> Implement systematic sampling to draw 10,000 for analysis. Then explain and
> demonstrate the main failure mode."*

In [ ]:
library(lubridate)

set.seed(42)

# ── Simulate time-ordered event log ──────────────────────────────────────────
N <- 1000000
df_log <- tibble::tibble(
  timestamp   = ymd_hms('2024-01-01 00:00:00') + seconds(0:(N-1)),
  response_ms = round(rexp(N, rate = 1/200), 1)
)

# ── Systematic sampling ───────────────────────────────────────────────────────
n     <- 10000
k     <- N %/% n                                  # step = 100, %/% is R's integer division
start <- sample(0:(k-1), 1)                        # random start in [0, k)
indices <- seq(start + 1, N, by = k)[1:n]           # +1 since R indexing is 1-based

sample_sys  <- df_log |> dplyr::slice(indices)
sample_base <- df_log$response_ms[indices]

cat(sprintf("Population : %s events\n", format(N, big.mark=",")))
cat(sprintf("Step k     : %d,  Random start: %d,  Sample n: %s\n", k, start, format(nrow(sample_sys), big.mark=",")))
cat(sprintf("Time range : %s -> %s\n", sample_sys$timestamp[1], sample_sys$timestamp[nrow(sample_sys)]))

# ── Confidence Interval ───────────────────────────────────────────────────────
x  <- sample_sys$response_ms
ci <- c(mean(x) - 1.96*sd(x)/sqrt(n), mean(x) + 1.96*sd(x)/sqrt(n))
cat(sprintf("\nSample mean : %.1f ms\n", mean(x)))
cat(sprintf("95%% CI      : (%.1f, %.1f) ms\n", ci[1], ci[2]))
cat(sprintf("True mean   : %.1f ms\n", mean(df_log$response_ms)))

# ── Failure mode: check autocorrelation at lag k ─────────────────────────────
autocorr <- acf(df_log$response_ms[1:(10*k)], lag.max = k, plot = FALSE)$acf[k+1]
cat(sprintf("\nAutocorrelation at lag k=%d: %.4f\n", k, autocorr))
cat("If |autocorr| > 0.1 consider SRS instead\n")

# ── Demonstrate: periodic data aligned with k ────────────────────────────────
periodic      <- sin(2 * pi * (0:(N-1)) / k) * 100 + 200
periodic_samp <- periodic[indices]
cat(sprintf("\nPeriodic signal -- true mean   : %.1f\n", mean(periodic)))
cat(sprintf("Periodic signal -- sample mean : %.1f  <- biased\n", mean(periodic_samp)))
cat(sprintf("Bias magnitude                 : %.1f\n", abs(mean(periodic_samp) - mean(periodic))))

**Common mistakes:**
- Not randomizing the starting point -- the sample is no longer probability-based
- Using `df[seq(1, N, k), ]` without a random start -- always starts at index 1
- Skipping the autocorrelation check -- periodic patterns (e.g. weekly seasonality in daily logs) can completely invalidate the sample. R's `acf()` is built into base R (the `stats` package, always loaded), no extra package needed

---
## SS4 -- Cluster Sampling

**Intuition:** When individuals are naturally grouped (sessions, users, stores),
sampling whole groups is often cheaper than sampling individuals. Select m
clusters at random and use all (or a sub-sample of) their members. The
statistical cost: units within a cluster are similar, so you get less
diversity per observation than SRS -- cluster SE > SRS SE for the same total n.

**Confidence Interval -- between-cluster variance:**

$$\bar{x}_{cl} = \frac{1}{m}\sum_i \bar{x}_i, \quad SE = \frac{s_{\bar{x}}}{\sqrt{m}}, \quad \text{CI} = \bar{x}_{cl} \pm t_{\alpha/2,\,m-1} \cdot SE$$

---

> **Interview prompt:** *"You have clickstream data across 500 app sessions.
> Sampling individual events is expensive. Implement cluster sampling by
> session and show numerically why the variance is higher than SRS of the
> same total n."*

In [ ]:
library(dplyr)

set.seed(42)

# ── Simulate session-level event data ─────────────────────────────────────────
n_sessions <- 500
sizes      <- sample(20:79, n_sessions, replace = TRUE)            # variable session size
session_ids <- rep(1:n_sessions, times = sizes)
df_events <- tibble::tibble(
  session_id = session_ids,
  duration_s = round(rexp(length(session_ids), rate = 1/30), 1),
  converted  = rbinom(length(session_ids), 1, 0.08)
)
cat(sprintf("Total events : %s across %d sessions\n", format(nrow(df_events), big.mark=","), n_sessions))

# ── One-stage: sample whole sessions ──────────────────────────────────────────
m        <- 50
selected <- sample(1:n_sessions, m, replace = FALSE)
one_stage <- df_events |> filter(session_id %in% selected)
cat(sprintf("\nOne-stage  : %d sessions, %s events\n", m, format(nrow(one_stage), big.mark=",")))

# ── Two-stage: subsample events within selected sessions ──────────────────────
parts <- list()
for (sid in selected) {
  grp <- df_events |> filter(session_id == sid)
  parts[[as.character(sid)]] <- grp |> slice_sample(prop = 0.5)
}
two_stage <- bind_rows(parts)
cat(sprintf("Two-stage  : %d sessions, %s events (50%% within each session)\n", m, format(nrow(two_stage), big.mark=",")))

# ── Confidence Interval (between-cluster variance) ────────────────────────────
cluster_means <- one_stage |> group_by(session_id) |> summarise(mean_dur = mean(duration_s), .groups = "drop")
c_bar  <- mean(cluster_means$mean_dur)
c_se   <- sd(cluster_means$mean_dur) / sqrt(m)
t_crit <- qt(0.975, df = m - 1)         # base R's qt() instead of a hard-coded constant
ci     <- c(c_bar - t_crit * c_se, c_bar + t_crit * c_se)

cat(sprintf("\nCluster mean : %.2f s\n", c_bar))
cat(sprintf("95%% CI       : (%.2f, %.2f) s\n", ci[1], ci[2]))
cat(sprintf("True mean    : %.2f s\n", mean(df_events$duration_s)))

# ── Variance comparison: cluster vs SRS of the same total n ──────────────────
srs_n <- nrow(one_stage)
srs_ests <- numeric(300); cluster_ests <- numeric(300)
for (i in 1:300) {
  srs_ests[i] <- mean(sample(df_events$duration_s, srs_n, replace = FALSE))
  sel <- sample(1:n_sessions, m, replace = FALSE)
  cm  <- df_events |> filter(session_id %in% sel) |> group_by(session_id) |> summarise(mean_dur = mean(duration_s), .groups = "drop")
  cluster_ests[i] <- mean(cm$mean_dur)
}

cat(sprintf("\nVariance comparison (300 resamples, same total n=%s):\n", format(srs_n, big.mark=",")))
cat(sprintf("  SRS SE     : %.4f\n", sd(srs_ests)))
cat(sprintf("  Cluster SE : %.4f  <- larger due to intra-session similarity\n", sd(cluster_ests)))
cat("  Trade-off  : cluster sampling is cheaper to collect but less statistically efficient\n")

**Cluster vs. Stratified -- the key distinction:**

| | Effect on variance | Within-group units are... | Use when |
| :--- | :--- | :--- | :--- |
| Stratified | Reduces variance | Heterogeneous (spread out) | Precision is the priority |
| Cluster | Increases variance | Homogeneous (similar) | Collection cost is the constraint |

**Common mistakes:**
- Confusing cluster and stratified -- they have **opposite** effects on variance
- Using only m = 2 or 3 clusters -- `qt(0.975, df=1)` = 12.7, giving an extremely wide CI; aim for m >= 10
- Forgetting that the CI uses **cluster-level means** as the unit, not individual events

---
## SS5 -- Bootstrap Resampling

**Intuition:** Your sample is the best proxy for the population you have. Draw
B samples *with replacement* from your data, compute the statistic on each --
the spread of those B estimates approximates the true sampling distribution.
Works for **any statistic**, requires no distributional assumptions, and is
the standard approach for model performance uncertainty in industry.

**Algorithm:** resample with replacement B times -> compute stat -> take percentiles.

**Confidence Interval (percentile method -- base R only):**

$$\text{CI} = [\hat{\theta}^*_{\alpha/2},\; \hat{\theta}^*_{1-\alpha/2}]$$

where $\hat{\theta}^*$ are the sorted bootstrap estimates and `quantile()` does the work
(R's direct analog to `np.percentile`, just expressed as a 0-1 probability instead of a 0-100 percentage).

---

> **Interview prompt (3 parts):**
> *(a) Implement a bootstrap CI for the median using base R only.*
> *(b) Use bootstrap to get a 95% CI on a classifier's test AUC.*
> *(c) Preview: bootstrap CI on the difference between two groups.*

In [ ]:
# ── (a) General bootstrap_ci: works for any statistic ────────────────────────
bootstrap_ci <- function(data, stat_fn, B = 2000, confidence = 0.95, seed = 42) {
  # Bootstrap confidence interval for any statistic.
  # stat_fn : function accepting a numeric vector, returning a scalar.
  # Uses the percentile method -- base R only, no packages needed.
  set.seed(seed)
  n <- length(data)
  boot_stats <- replicate(B, stat_fn(sample(data, n, replace = TRUE)))
  alpha <- 1 - confidence
  ci <- quantile(boot_stats, c(alpha/2, 1 - alpha/2))
  list(lo = ci[1], hi = ci[2], boot_stats = boot_stats)
}

set.seed(42)
data <- rexp(300, rate = 1/50)    # skewed -- bootstrap shines here
B <- 2000

stats_to_test <- list(
  mean        = mean,
  median      = median,
  std         = sd,
  `90th pctile` = function(x) quantile(x, 0.9)
)

cat(sprintf("%-14s %10s %26s  %14s\n", "Statistic", "Observed", "95% CI", "Bootstrap SE"))
cat(strrep("-", 68), "\n")
for (name in names(stats_to_test)) {
  fn  <- stats_to_test[[name]]
  obs <- fn(data)
  res <- bootstrap_ci(data, fn, B = B)
  cat(sprintf("%-14s %10.3f   (%.3f, %.3f)  %14.3f\n", name, obs, res$lo, res$hi, sd(res$boot_stats)))
}

In [ ]:
# ── (b) Bootstrap CI on model evaluation metrics ──────────────────────────────
set.seed(42)
n <- 1000
y_true <- rbinom(n, 1, 0.3)
y_prob <- pmin(pmax(y_true * 0.6 + rnorm(n, 0, 0.25), 0), 1)
y_pred <- as.integer(y_prob >= 0.5)

accuracy <- function(yt, yp) mean(yt == yp)

roc_auc <- function(yt, yp) {
  # Trapezoidal AUC (base R only, no package needed)
  ord  <- order(yp, decreasing = TRUE)
  yt_s <- yt[ord]
  pos  <- sum(yt); neg <- sum(1 - yt)
  tp <- cumsum(yt_s) / pos
  fp <- cumsum(1 - yt_s) / neg
  tp <- c(0, tp); fp <- c(0, fp)
  sum(diff(fp) * (tp[-1] + tp[-length(tp)]) / 2)   # manual trapezoid
}

bootstrap_metric_ci <- function(y_true, y_score, metric_fn, B = 2000, confidence = 0.95, seed = 42) {
  # Bootstrap CI for any model metric. metric_fn(y_true, y_score) -> scalar.
  set.seed(seed)
  n <- length(y_true)
  boot <- replicate(B, {
    idx <- sample(1:n, n, replace = TRUE)
    metric_fn(y_true[idx], y_score[idx])
  })
  alpha <- 1 - confidence
  ci <- quantile(boot, c(alpha/2, 1 - alpha/2))
  list(point = metric_fn(y_true, y_score), lo = ci[1], hi = ci[2], boot = boot)
}

acc_res <- bootstrap_metric_ci(y_true, y_pred, accuracy)
auc_res <- bootstrap_metric_ci(y_true, y_prob, roc_auc)

cat("Bootstrap CI on model performance  (n=1000, B=2000)\n")
cat(sprintf("  %-12s %10s %26s\n", "Metric", "Point est", "95% CI"))
cat(sprintf("  %-12s %10.4f   (%.4f, %.4f)\n", "Accuracy", acc_res$point, acc_res$lo, acc_res$hi))
cat(sprintf("  %-12s %10.4f   (%.4f, %.4f)\n", "AUC", auc_res$point, auc_res$lo, auc_res$hi))
cat("\nThe CI shows the uncertainty in the metric across hypothetical test sets.\n")
cat("A single test-set score is just one draw from this distribution.\n")

In [ ]:
# ── (c) Bootstrap CI on the difference (preview of SS9) ───────────────────────
set.seed(42)
group_a <- rnorm(300, 50, 10)    # control
group_b <- rnorm(300, 53, 10)    # treatment (+3 units)

bootstrap_diff_ci <- function(a, b, stat_fn = mean, B = 2000, confidence = 0.95, seed = 42) {
  # Bootstrap CI for stat_fn(b) - stat_fn(a).
  set.seed(seed)
  diffs <- replicate(B,
    stat_fn(sample(b, length(b), replace = TRUE)) - stat_fn(sample(a, length(a), replace = TRUE))
  )
  alpha <- 1 - confidence
  ci <- quantile(diffs, c(alpha/2, 1 - alpha/2))
  list(obs = stat_fn(b) - stat_fn(a), lo = ci[1], hi = ci[2])
}

res <- bootstrap_diff_ci(group_a, group_b)
cat(sprintf("Observed difference (B - A) : %.3f\n", res$obs))
cat(sprintf("95%% CI on difference        : (%.3f, %.3f)\n", res$lo, res$hi))
cat(sprintf("Excludes zero (significant) : %s\n", res$lo > 0))
cat("\nFull two-sample test with p-value and permutation alternative in SS9.\n")

**Bootstrap CI methods compared:**

| Method | Package needed? | Handles skew? | When to use |
| :--- | :--- | :--- | :--- |
| Percentile | No -- `quantile()` (base R) | No | Quick CI in interviews |
| Basic / pivot | No | Partially | Better for biased bootstrap |
| BCa | Yes (the `boot` package) | Yes | Production code |

**Common mistakes:**
- B < 1000 resamples -- CIs are unstable; B = 2000 is the minimum for interviews
- Bootstrapping time-series with plain row resampling -- violates independence; use block bootstrap (e.g. `tsbootstrap()` from the `boot`/`tseries` packages)
- Using the same seed for both groups in a two-sample bootstrap -- they must be sampled independently. In R, `set.seed()` is called **once** before the replicate loop precisely because it controls a single global stream; calling `set.seed()` again mid-loop would make every resample identical, a distinctly R-flavored version of this mistake

---
## SS6 -- Jackknife Resampling

**Intuition:** Leave one observation out at a time, recompute the statistic on
the remaining n-1 -- repeat for every observation. The n leave-one-out
estimates reveal how much each individual observation influences the result.
Use jackknife for (a) bias correction and (b) finding influential
observations. Only works reliably for **smooth** statistics (mean, ratio) --
fails for median and quantiles. Bootstrap is the safer default; jackknife is
the specialist tool.

**Jackknife bias correction:**

$$\text{bias} = (n-1)(\bar{\theta}_{(.)} - \hat{\theta}), \quad \hat{\theta}_{jack} = n\hat{\theta} - (n-1)\bar{\theta}_{(.)}$$

**Confidence Interval:**

$$SE_{jack} = \sqrt{\frac{n-1}{n}\sum_i(\hat{\theta}_{(i)} - \bar{\theta}_{(.)})^2}, \quad \text{CI} = \hat{\theta} \pm t_{\alpha/2,n-1} \cdot SE_{jack}$$

---

> **Interview prompt:** *"Implement leave-one-out jackknife to identify the
> observations that most influence the sample mean. Then show why jackknife
> gives wrong answers for the median."*

In [ ]:
set.seed(42)

# Dataset with known outliers to make influential-obs detection interesting
data <- c(rnorm(95, 50, 10), c(90, 85, 80, 15, 20))
n <- length(data)

# ── Jackknife leave-one-out estimates ────────────────────────────────────────
jk_stats <- sapply(1:n, function(i) mean(data[-i]))    # base R: data[-i] drops index i -- no separate np.delete needed

# ── Bias estimation and correction ───────────────────────────────────────────
theta_hat  <- mean(data)
theta_bar  <- mean(jk_stats)
bias_est   <- (n - 1) * (theta_bar - theta_hat)
theta_jack <- n * theta_hat - (n - 1) * theta_bar       # bias-corrected

# ── Jackknife SE and CI ───────────────────────────────────────────────────────
jk_var <- ((n - 1) / n) * sum((jk_stats - theta_bar)^2)
jk_se  <- sqrt(jk_var)
t_crit <- qt(0.975, df = n - 1)                          # base R's exact qt(), no hard-coded constant needed
ci     <- c(theta_hat - t_crit * jk_se, theta_hat + t_crit * jk_se)

cat(sprintf("Observed mean       : %.4f\n", theta_hat))
cat(sprintf("Jackknife bias      : %.6f  (small = unbiased estimator)\n", bias_est))
cat(sprintf("Bias-corrected mean : %.4f\n", theta_jack))
cat(sprintf("Jackknife SE        : %.4f\n", jk_se))
cat(sprintf("95%% CI              : (%.3f, %.3f)\n", ci[1], ci[2]))

# ── Interview task: detect most influential observations ──────────────────────
influence <- abs(jk_stats - theta_hat)                   # per-obs influence
top_k <- 5
top_idx <- order(influence, decreasing = TRUE)[1:top_k]

cat(sprintf("\nTop %d most influential observations:\n", top_k))
cat(sprintf("  %-6s %6s %8s %10s %12s\n", "Rank", "Index", "Value", "LOO mean", "Influence"))
for (rank in 1:top_k) {
  idx <- top_idx[rank]
  cat(sprintf("  %-6d %6d %8.1f %10.4f %12.4f\n", rank, idx, data[idx], jk_stats[idx], influence[idx]))
}

# ── When jackknife fails: the median ─────────────────────────────────────────
jk_medians  <- sapply(1:n, function(i) median(data[-i]))
jk_se_med   <- sqrt(((n-1)/n) * sum((jk_medians - mean(jk_medians))^2))
boot_se_med <- sd(replicate(2000, median(sample(data, n, replace = TRUE))))
cat(sprintf("\nJackknife SE for median : %.4f  <- wrong (non-smooth statistic)\n", jk_se_med))
cat(sprintf("Bootstrap SE for median : %.4f  <- correct\n", boot_se_med))
cat("Rule: use jackknife for mean/ratio; use bootstrap for median/quantile/custom metrics\n")

**Jackknife vs. Bootstrap:**

| | Resamples | Smooth stats only? | Bias correction? | Cost |
| :--- | :--- | :--- | :--- | :--- |
| Jackknife | n (exact) | Yes | Yes | O(n) |
| Bootstrap | B >= 2000 | No | Partial (BCa) | O(B * n) |

**Common mistakes:**
- Applying jackknife to the median, quantiles, or max -- produces wrong SE estimates
- Confusing jackknife with LOOCV -- same leave-one-out structure, different purpose (LOOCV measures prediction error; jackknife measures statistical variance)
- Interpreting high jackknife influence as "delete this observation" -- it means investigate it, not remove it

---
## SS7 -- Weighted Sampling

**Intuition:** When observations have unequal importance -- due to class
imbalance, oversampling, or survey design -- weights restore
representativeness. Each observation gets weight proportional to 1 /
(selection probability). In DS interviews this almost always appears as
**class imbalance**: the minority class needs more weight so the model does
not ignore it.

**Inverse class frequency weight:** $w_i = 1 / \text{count}(\text{class of } i)$

**Confidence Interval using Kish effective sample size:**

$$\bar{x}_w = \frac{\sum w_i x_i}{\sum w_i}, \quad n_{eff} = \frac{(\sum w_i)^2}{\sum w_i^2}, \quad SE_w = \sqrt{\frac{s_w^2}{n_{eff}}}$$

Using $n_{eff}$ instead of raw $n$ correctly accounts for high variance in weights.

---

> **Interview prompt:** *"You have a binary dataset with 95/5 class imbalance.
> Implement (a) weighted sampling to create a balanced training set, and (b)
> inverse-frequency weighting to compute a balanced accuracy metric. Verify
> the class balance after sampling."*

In [ ]:
library(dplyr)

set.seed(42)

# ── Simulate imbalanced binary dataset ───────────────────────────────────────
N      <- 5000
labels <- sample(c(0, 1), N, replace = TRUE, prob = c(0.95, 0.05))
df <- tibble::tibble(
  feature = rnorm(N, mean = labels * 0.5, sd = 1),
  label   = labels
)
cat("Original class distribution:\n")
print(table(df$label))

# ── Inverse class frequency weights ──────────────────────────────────────────
class_counts <- table(df$label)
df <- df |> mutate(sample_weight = 1.0 / as.numeric(class_counts[as.character(label)]))

# ── (a) Weighted sampling to balance training set ────────────────────────────
# replace=TRUE required when minority class is smaller than requested n
balanced_dplyr <- df |> slice_sample(n = 1000, weight_by = sample_weight, replace = TRUE)

# Base R version (weights must sum to 1 explicitly -- sample() normalizes
# automatically too, but being explicit here mirrors the NumPy-vs-pandas split
# in the source: dplyr normalizes for you, base R's sample(prob=) also
# normalizes for you, so this distinction is actually less of a footgun in R)
idx_base <- sample(1:nrow(df), size = 1000, replace = TRUE, prob = df$sample_weight)
balanced_base <- df[idx_base, ]

# ── Verify class balance ──────────────────────────────────────────────────────
cat("\nBalanced sample proportions (dplyr):\n")
print(round(prop.table(table(balanced_dplyr$label)), 3))
cat("\nBalanced sample proportions (base R):\n")
print(round(prop.table(table(balanced_base$label)), 3))

# ── (b) Inverse-frequency weighted accuracy ───────────────────────────────────
y_true <- df$label
y_pred <- as.integer(df$feature > 0.25)        # simple threshold
w      <- df$sample_weight

unweighted_acc <- mean(y_true == y_pred)
weighted_acc   <- weighted.mean(y_true == y_pred, w)    # base R's weighted.mean() -- balanced accuracy

cat(sprintf("\nUnweighted accuracy : %.4f  <- dominated by majority class (easy 0s)\n", unweighted_acc))
cat(sprintf("Weighted accuracy   : %.4f  <- balanced; penalises minority errors equally\n", weighted_acc))

# ── Confidence Interval using Kish effective n ───────────────────────────────
vals   <- df$feature
w_mean <- weighted.mean(vals, w)
w_var  <- weighted.mean((vals - w_mean)^2, w)
n_eff  <- sum(w)^2 / sum(w^2)                            # Kish formula
w_se   <- sqrt(w_var / n_eff)
ci     <- c(w_mean - 1.96 * w_se, w_mean + 1.96 * w_se)

cat(sprintf("\nKish effective n    : %.0f  (vs raw n = %d)\n", n_eff, N))
cat(sprintf("Weighted mean       : %.4f\n", w_mean))
cat(sprintf("95%% CI              : (%.4f, %.4f)\n", ci[1], ci[2]))
cat("Using n_eff prevents overconfident CIs when weights are highly variable\n")

**Common mistakes:**
- `sample(prob = w, replace = FALSE)` fails (or is badly inefficient) when weights are very unequal and the sample is large -- use `replace = TRUE` for oversampling, same logic as pandas
- Both `dplyr::slice_sample(weight_by=)` and base R's `sample(prob=)` normalize weights automatically — unlike the Python source where NumPy specifically requires pre-normalized weights and pandas does not, **R's two main paths agree with each other**, removing that particular footgun entirely
- Using raw n in the weighted SE formula -- always use Kish n_eff; otherwise the CI is too narrow when weights vary a lot
- Oversampling the test set -- only oversample the training set; evaluate on the natural class distribution

---
## SS8 -- Stratified K-Fold Cross-Validation

**Intuition:** K-fold splits the data into k equal folds, trains on k-1 folds,
and validates on the remaining one -- rotating until every fold has been the
validation set. **Stratified** k-fold ensures each fold has the same class
proportions as the full dataset. Without stratification, a small minority
class can vanish entirely from a validation fold, making the CV score
meaningless for that class.

**Why this appears in DS interviews:** It is the most common cross-validation
pattern in production ML, yet packages like tidymodels' `rsample::vfold_cv()`
hide all the logic. Interviewers use "implement it from scratch" to test
whether you understand what it is actually doing.

**Core logic:** Assign fold IDs round-robin **within each class**, then rotate.

---

> **Interview prompt:** *"Implement stratified k-fold cross-validation without
> a modeling package. For every fold, verify the class proportions in the
> validation set match the full dataset. Show what goes wrong without
> stratification on an imbalanced dataset."*

In [ ]:
library(dplyr)

set.seed(42)

# ── Simulate imbalanced 3-class dataset ───────────────────────────────────────
N      <- 1000
labels <- sample(c(0, 1, 2), N, replace = TRUE, prob = c(0.70, 0.20, 0.10))
df <- as.data.frame(matrix(rnorm(N * 3), ncol = 3, dimnames = list(NULL, c('f1','f2','f3'))))
df$label <- labels
pop_props <- prop.table(table(df$label))
cat("Population class proportions:\n")
print(round(pop_props, 3))

# ── Stratified K-Fold from scratch ────────────────────────────────────────────
stratified_kfold <- function(df, target_col, k = 5, seed = 42) {
  # Returns a list of (train_indices, val_indices) for k stratified folds.
  # Core idea: assign fold IDs round-robin within each class,
  # then rotate which fold is the validation set.
  set.seed(seed)
  fold_ids <- rep(-1L, nrow(df))
  for (lbl in unique(df[[target_col]])) {
    idx <- which(df[[target_col]] == lbl)
    idx <- sample(idx)                                  # shuffle within class
    fold_ids[idx] <- (seq_along(idx) - 1) %% k           # assign round-robin
  }
  folds <- list()
  for (fold in 0:(k-1)) {
    val_idx   <- which(fold_ids == fold)
    train_idx <- which(fold_ids != fold)
    folds[[fold + 1]] <- list(train = train_idx, val = val_idx)
  }
  folds
}

# ── Run and verify proportions every fold ─────────────────────────────────────
k <- 5
folds <- stratified_kfold(df, 'label', k = k)
cat(sprintf("\nStratified %d-Fold -- validation class proportions:\n", k))
cat(sprintf("%-6s %8s %6s  %8s %8s %8s\n", "Fold", "Train", "Val", "cls_0", "cls_1", "cls_2"))
cat(strrep("-", 50), "\n")
for (fold in seq_along(folds)) {
  train_idx <- folds[[fold]]$train; val_idx <- folds[[fold]]$val
  vp <- prop.table(table(factor(df$label[val_idx], levels = c(0,1,2))))
  cat(sprintf("%-6d %8d %6d  %8.3f %8.3f %8.3f\n", fold - 1, length(train_idx), length(val_idx), vp[1], vp[2], vp[3]))
}
cat(sprintf("%-6s %8s %6s  %8.3f %8.3f %8.3f\n", "Pop", "", "", pop_props[1], pop_props[2], pop_props[3]))

# ── What goes wrong without stratification ─────────────────────────────────────
cat(sprintf("\nNon-stratified %d-Fold -- class_2 proportion per fold (target %.3f):\n", k, pop_props[3]))
all_idx   <- sample(1:N)
fold_size <- N %/% k
for (fold in 0:(k-1)) {
  val_idx <- all_idx[(fold*fold_size + 1):((fold+1)*fold_size)]
  vp    <- prop.table(table(factor(df$label[val_idx], levels = c(0,1,2))))
  prop2 <- vp[3]
  flag  <- if (abs(prop2 - pop_props[3]) > 0.02) "  <- off target" else ""
  cat(sprintf("  Fold %d: class_2 = %.3f%s\n", fold, prop2, flag))
}

**Common mistakes:**
- Shuffling the full dataset once and then slicing -- does NOT stratify; the shuffle mixes classes but does not control fold composition
- Data leakage: applying preprocessing (scaling, encoding -- see `DM_Advanced` §recipes) on the full dataset before splitting -- always `prep()` transformers on the training fold only
- Reaching for `recipes`/`rsample` machinery before understanding the round-robin logic above -- once you do, `rsample::vfold_cv(df, v = k, strata = label)` does exactly this in production code
- Using the same fold split for both hyperparameter tuning and final evaluation -- use nested CV or a held-out test set

---
## SS9 -- Two-Sample Bootstrap & Permutation Test

**Intuition (bootstrap):** To get a CI on the *difference* between two groups,
resample each group independently with replacement, compute the difference of
the statistic on each pair of resamples -- the distribution of those
differences is the bootstrap sampling distribution of the difference.

**Intuition (permutation test):** Under the null hypothesis that the two
groups are exchangeable, shuffling which observations are labelled A vs B
should produce a difference close to zero. Compute this shuffle-difference B
times to build the null distribution, then check where the observed
difference sits.

**CI on the difference:**

$$\text{CI} = [\delta^*_{\alpha/2},\; \delta^*_{1-\alpha/2}] \quad \text{where } \delta^* = \hat{\theta}_B^* - \hat{\theta}_A^*$$

**Bootstrap p-value (one-sided):** $p = P(\delta^* \leq 0)$

**Permutation p-value (two-sided):** $p = P(|\delta_{null}| \geq |\delta_{obs}|)$

---

> **Interview prompt:** *"You ran an A/B test. Group A has 500 users (8%
> conversion), Group B has 500 users. Without a stats package, determine
> whether the difference in conversion rates is statistically significant.
> Report a 95% CI on the difference and a p-value."*

In [ ]:
set.seed(42)

# ── A/B test: binary conversion data ─────────────────────────────────────────
n_a <- 500; n_b <- 500
conv_a   <- rbinom(n_a, 1, 0.08)    # control:   8% rate
conv_b   <- rbinom(n_b, 1, 0.11)    # treatment: 11% rate
obs_diff <- mean(conv_b) - mean(conv_a)

cat(sprintf("Group A conversion : %.4f  (n=%d)\n", mean(conv_a), n_a))
cat(sprintf("Group B conversion : %.4f  (n=%d)\n", mean(conv_b), n_b))
cat(sprintf("Observed diff B-A  : %+.4f\n", obs_diff))

# ── Bootstrap CI on the difference ───────────────────────────────────────────
bootstrap_diff_ci <- function(a, b, stat_fn = mean, B = 10000, confidence = 0.95, seed = 42) {
  # Bootstrap CI for stat_fn(b) - stat_fn(a).
  # Resamples each group independently.
  set.seed(seed)
  na <- length(a); nb <- length(b)
  boot_d <- sapply(1:B, function(i) {
    stat_fn(b[sample(1:nb, nb, replace = TRUE)]) - stat_fn(a[sample(1:na, na, replace = TRUE)])
  })
  alpha <- 1 - confidence
  ci <- quantile(boot_d, c(alpha/2, 1 - alpha/2))
  p_one <- mean(boot_d <= 0)            # P(B <= A), one-sided
  list(lo = ci[1], hi = ci[2], p_one = p_one, boot_d = boot_d)
}

bres <- bootstrap_diff_ci(conv_a, conv_b, B = 10000)
cat(sprintf("\n95%% CI on difference : (%.4f, %.4f)\n", bres$lo, bres$hi))
cat(sprintf("CI excludes zero     : %s  (significant at 5%%)\n", bres$lo > 0))
cat(sprintf("Bootstrap p (one-sided): %.4f\n", bres$p_one))

# ── Permutation test (two-sided, exact null) ──────────────────────────────────
permutation_test <- function(a, b, B = 10000, seed = 42) {
  # Tests H0: mean(a) == mean(b) by permuting group labels.
  # Returns two-sided p-value and the null distribution.
  set.seed(seed)
  combined <- c(a, b)
  n_a <- length(a); n_total <- length(combined)
  obs <- mean(b) - mean(a)
  null_d <- sapply(1:B, function(i) {
    perm <- sample(combined)
    mean(perm[(n_a+1):n_total]) - mean(perm[1:n_a])
  })
  p_two <- mean(abs(null_d) >= abs(obs))
  list(p_two = p_two, null_d = null_d)
}

pres <- permutation_test(conv_a, conv_b, B = 10000)
cat(sprintf("Permutation p (two-sided): %.4f\n", pres$p_two))

# ── Side-by-side comparison ───────────────────────────────────────────────────
cat("\n", strrep("=", 60), "\n", sep = "")
cat(sprintf("%-28s %s\n", "Method", "Result"))
cat(strrep("=", 60), "\n")
cat(sprintf("%-28s (%.4f, %.4f)\n", "Bootstrap CI (95%)", bres$lo, bres$hi))
cat(sprintf("%-28s %.4f\n", "Bootstrap p (one-sided)", bres$p_one))
cat(sprintf("%-28s %.4f\n", "Permutation p (two-sided)", pres$p_two))
cat(strrep("=", 60), "\n")
cat("\nBootstrap   --> use when you need the EFFECT SIZE and its uncertainty\n")
cat("Permutation --> use when you need a p-value under an exact null hypothesis\n")

**Bootstrap vs. Permutation -- when to use each:**

| | Bootstrap CI | Permutation test |
| :--- | :--- | :--- |
| Primary output | Effect size + uncertainty | p-value |
| Null hypothesis | Not required | Exact exchangeability |
| Works for any statistic | Yes | Yes |
| Best for | Estimation, effect size reporting | Hypothesis testing |

**Common mistakes:**
- Using a one-sided p-value when the test is not directional -- report two-sided unless the alternative was specified beforehand
- Reporting only a p-value without a CI -- the CI tells you whether the effect is practically meaningful, not just statistically significant
- Resampling both groups from the combined pool in the bootstrap -- groups must be resampled **independently** (combine-and-resample is the permutation test, not the bootstrap)
- Too few permutations -- B = 10,000 gives a p-value stable to ~0.003; B = 1,000 is too noisy for a final answer

---
## SS10 -- Interview Decision Guide

```
What is the interviewer asking you to do?

+-- "Split data" / "train-test"
|   +-- Plain random split                          -> SS1 train_test_split
|   +-- "Preserve class balance" / imbalanced data  -> SS2 stratified_split
|   +-- "Large ordered data" / log files            -> SS3 Systematic sampling
|   +-- "Grouped data" / sessions / users           -> SS4 Cluster sampling
|
+-- "Cross-validation" / "k-fold"
|   +-- Any mention of class imbalance              -> SS8 Stratified k-fold
|
+-- "Confidence interval" / "uncertainty"
|   +-- CI for a standard stat, large n             -> z = 1.96 approximation
|   +-- CI for any stat, unknown distribution       -> SS5 Bootstrap (quantile())
|   +-- "Which obs are influential?" / bias         -> SS6 Jackknife
|   +-- Imbalanced / survey data                    -> SS7 Weighted CI (Kish n_eff)
|
+-- "A/B test" / "compare two groups" / "significance"
    +-- Need a CI on the difference                 -> SS9 Bootstrap diff CI
    +-- Need a p-value                              -> SS9 Permutation test
    +-- Need both                                   -> SS9 Run both, report both
```

**Full comparison:**

| Method | Core R pattern | CI approach | Extra package? |
| :--- | :--- | :--- | :--- |
| SRS | `slice_sample(prop = 1)` + slice | z * se | No |
| Stratified split | loop over groups + `slice_sample` | Weighted stratum variance | No |
| Systematic | `df |> slice(seq(start, N, k))` | ~SRS z-approx | No |
| Cluster | `filter(id %in% selected)` | Between-cluster SE + t (`qt()`) | No |
| Bootstrap | `replicate(B, sample(x, n, TRUE))` + `quantile()` | `quantile()` | No |
| Jackknife | `data[-i]` per i | Jackknife SE + t (`qt()`) | No |
| Weighted | `slice_sample(weight_by = col)` | Weighted SE + Kish n_eff | No |
| Stratified k-fold | loop over groups + round-robin fold IDs | Not applicable | No |
| Two-sample bootstrap | Independent resample each group | `quantile()` on diffs | No |
| Permutation test | `sample()` to shuffle labels | p-value from null dist | No |